# 📊 Performance Metric Governance Lab (Section II-D)

This notebook implements the **Quantitative 'Research Loop'** for **Section II-D (Performance Metrics)**.
It combines **Regex Mining** (for coverage) with **Groq LLM Verification** (for semantic precision) to enforce strict governance on performance metrics (e.g., distinguishing *Resolution* vs *Accuracy*, *OSNR* vs *ESNR*).

## Research Objectives (Section II-D)
1. **Detection**: Identify performance metrics used in O-ISAC papers (Comm, Sensing, Joint).
2. **Governance Check**: Validate canonical taxonomy (Symbols, Units, Plane Definitions).
3. **Evidence Extraction**: Extract precise quotes and locators for the manuscript.

## Target Scope (II-D)
- **Communication**: BER, SER, Capacity, Throughput, EVM.
- **Sensing (Estimation)**: Range/Angle/Velocity Resolution ($Δ$), Accuracy ($σ$, RMSE), CRB.
- **Sensing (Detection)**: $P_d$, $P_{fa}$, ROC Curves.
- **Joint/Tradeoff**: CRQ, Pareto Frontiers, Radar-Comm Tradeoffs.
- **Signal Quality**: OSNR (Optical), SNR/ESNR (Electrical), SINR.


In [ ]:
# @title 1. Install & Setup
!pip install -q groq

from google.colab import drive
import os
import glob
import json
import csv
import re
from collections import Counter
from groq import Groq
from google.colab import userdata

# 1.1 Mount Drive & Set Base Dir
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"

if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"✅ Working Directory set to: {os.getcwd()}")
else:
    print(f"❌ Path not found: {BASE_DIR}. Please check your Drive structure.")

# 1.2 Load API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    print("🔑 Groq API Key loaded.")
except Exception as e:
    print(f"⚠️ Error: {e}. Ensure 'GROQ_API_KEY' is in Colab Secrets.")

# 1.3 Define Output Path
OUTPUT_DIR = "analysis/II_evidence_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVIDENCE_CSV = os.path.join(OUTPUT_DIR, "section2D_evidence.csv")
GOVERNANCE_DOC = "analysis/II_metric_evaluation_governance_2D.md"

In [ ]:
# @title 2. Data Loader (Recursive)
def load_processed_markdowns(target_ids=None, limit=None):
    """
    Loads markdown files recursively from 'data/processed_markdowns'.
    Matches the logic of previous Governance Labs.
    """
    search_path = os.path.join("data", "processed_markdowns")
    
    # Recursive search for all .md files
    all_files = glob.glob(os.path.join(search_path, "**", "*.md"), recursive=True)
    
    # Filter for O_ISAC or COMST files
    valid_files = [f for f in all_files if "O_ISAC" in f or "COMST" in f]
    
    # ID Filtering
    selected_files = []
    if target_ids:
        print(f"Applying filter for {len(target_ids)} Target IDs...")
        for f_path in valid_files:
            p_id = os.path.basename(f_path).replace('.md', '')
            if p_id in target_ids:
                selected_files.append(f_path)
    else:
        selected_files = valid_files

    if limit:
        selected_files = selected_files[:limit]
        
    print(f"Found {len(valid_files)} total files. Loading {len(selected_files)} for analysis.")
    
    data = []
    for f_path in selected_files:
        p_id = os.path.basename(f_path).replace('.md', '')
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                content = f.read()
                data.append((p_id, content))
        except Exception as e:
            print(f"Error reading {f_path}: {e}")
            
    return data

In [ ]:
# @title 3. Define II-D Research Agent (Metric Extraction)
def analyze_metric_governance(paper_text, paper_id):
    """
    Asks LLM to extract Performance Metrics from the paper.
    Includes 'Reasoning' for Evidence Locking and Plane Distinction.
    """
    
    system_prompt = """
    You are a Senior Metrologist auditing O-ISAC papers for Section II-D (Performance Metrics).
    Your goal is to extract EXACT evidence of usage for:
    1. **Comm Metrics**: BER, Capacity, Throughput, EVM.
    2. **Sensing (Estimation)**: Resolution (Delta), Accuracy (RMSE/CRB).
    3. **Sensing (Detection)**: Pd (Detect Prob), Pfa (False Alarm), ROC.
    4. **Signal Quality**: OSNR (Optical Plane) vs SNR (Electrical Plane).
    
    # CRITICAL RULES:
    - Distinguish 'Resolution' (separation cap) vs 'Accuracy' (error variance).
    - Capture units if present (e.g. 'bps/Hz', 'm', 'dB').
    - For SNR/OSNR, determine the PLANE (Optical vs Electrical) if context allows.

    # OUTPUT FORMAT:
    Return a JSON object with a 'reasoning' string and a 'findings' list.
    Example:
    {
      "reasoning": "The authors evaluate sensing performance using CRB for range estimation (Eq 12) and compare it with RMSE. For comms, they plot BER vs SNR.",
      "findings": [
         { "category": "Sensing (Est)", "metric": "CRB", "symbol": "CRB", "plane": "N/A", "evidence": "derive the CRB for range estimation..." },
         { "category": "Comm", "metric": "BER", "symbol": "Pb", "plane": "Electrical", "evidence": "BER performance against electrical SNR..." }
      ]
    }
    """
    
    user_prompt = f"""
    Paper ID: {paper_id}
    
    Analyze this text strictly. DO NOT hallucinate. 
    
    Text Content (First 35k chars): 
    {paper_text[:35000]}
    """
    
    try:
        completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        return {"error": str(e), "reasoning": "Fail", "findings": []}


In [ ]:
# @title 4. Execute Research Loop (Targeted)

# ==========================================
# CONFIGURATION
# ==========================================
# Recommended Papers for Metric Diversity
TARGET_PAPERS = ['O_ISAC_029', 'O_ISAC_001', 'O_ISAC_061', 'O_ISAC_199', 'O_ISAC_005']
LIMIT = None 
OUTPUT_CSV = "analysis/II_evidence_v2/section2D_evidence_LLM.csv"
# ==========================================

# 1. Load Papers
papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)

# 2. Run Agent
all_findings = []
print(f"\n🚀 Starting II-D Metric Analysis on {len(papers)} papers...\n")

for pid, text in papers:
    print(f"Processing {pid}...")
    result = analyze_metric_governance(text, pid)
    
    reasoning = result.get("reasoning", "No reasoning provided.")
    findings = result.get("findings", [])
    
    print(f"  🧠 Agent Reasoning: {reasoning[:150]}...")
    
    for f in findings:
        f["paper_id"] = pid
        f["full_reasoning"] = reasoning
        all_findings.append(f)
        print(f"     -> 📈 Found {f.get('category')}: {f.get('metric')}")
    print("-"*40)

# 3. Export Results
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
if all_findings:
    keys = ["paper_id", "category", "metric", "symbol", "plane", "evidence", "full_reasoning"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_findings)
    print(f"\n✅ Saved {len(all_findings)} confirmed evidence rows to {OUTPUT_CSV}")
    # Show first few results
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV).head())
else:
    print("\n⚠️ No findings extracted.")

In [ ]:
# @title 5. Generate Governance Artifacts (II-D)
def generate_governance_doc(findings):
    if not findings: return
    
    metrics = [f['metric'] for f in findings]
    counts = Counter(metrics)
    
    content = "# Section II-D: Performance Metric Governance\n\n"
    content += "## 1. Metric Catalog (Detected Usage)\n"
    for m, c in counts.most_common():
        content += f"- **{m}**: {c} occurrences\n"
        
    content += "\n## 2. Symbol Conventions (Consensus)\n"
    content += "- **Resolution**: $\\Delta r$ (Range), $\\Delta v$ (Velocity)\n"
    content += "- **Accuracy**: $\\sigma$ (RMSE), CRB (Lower Bound)\n"
    content += "- **Detection**: $P_d$ (Prob Detection), $P_{fa}$ (False Alarm)\n"
    
    with open(GOVERNANCE_DOC, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Generated {GOVERNANCE_DOC}")

if all_findings:
    generate_governance_doc(all_findings)